<a href="https://colab.research.google.com/github/brandy99swords/AIML2003_NLP/blob/main/week2%20/%20lab1_factcheck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai -q

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello to AIML 2003!"
)
print(response.text)

Hello, AIML 2003!

Greetings to a foundational year in the history of Artificial Intelligence Markup Language! That was a time when AIML was a prominent and exciting way to create conversational agents and chatbots, like the famous ALICE.

It represents a key era for rule-based AI, early natural language processing, and the development of accessible tools for building interactive dialogue systems.

Ready for some `<category>` and `<template>` fun!


In [ ]:
# Your claims dataset
claims = [
    {"text": "GPT-4 was trained using RLHF.",
     "truth": "true"},
    {"text": "Adobe's Firefly AI image generator was exclusively trained on content from Adobe Stock and public domain images.",
     "truth": "true"},
    {"text": "Canva's AI image generation features solely use publicly licensed data for training to avoid copyright issues.",
     "truth": "half-true"},
    {"text": "Midjourney's AI models are trained only on images explicitly licensed from artists, without any web scraping.",
     "truth": "false"},
    {"text": "Google's Imagen AI models are trained entirely on internal datasets, never incorporating any internet-scraped content.",
     "truth": "false"},
    {"text": "Companies like Stability AI (Stable Diffusion) have publicly acknowledged using large datasets that include scraped internet images.",
     "truth": "true"},
    {"text": "Most major AI image generators explicitly state that their training data *does not* include any copyrighted material from the open web.",
     "truth": "false"},
    {"text": "Private libraries of licensed assets are generally preferred by enterprise AI image providers to mitigate legal risks.",
     "truth": "true"},
    {"text": "The ethics of using internet-scraped data for AI training are universally agreed upon by major tech companies and artists.",
     "truth": "false"},
    {"text": "Adobe guarantees that Firefly users will not face copyright infringement claims for images generated with its tool.",
     "truth": "true"},
    {"text": "OpenAI's DALL-E 3, integrated with ChatGPT, strictly uses Creative Commons licensed images for its training data.",
     "truth": "half-true"}
]

In [ ]:
# Approach 1 - Zero-shot
def verdict_zero_shot(claim):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Is this claim true, false, or half-true? "
                 f"Respond with only the verdict.\n\nClaim: {claim}"
    )
    return response.text.strip().lower()

zero_shot = [verdict_zero_shot(c["text"]) for c in claims]
print("Zero-shot results:", zero_shot)

Zero-shot results: ['true', 'false', 'false', 'false', 'false', 'true', 'false', 'true', 'false', 'half-true', 'false']


In [ ]:
# Approach 2 - Few-shot
def verdict_few_shot(claim):
    prompt = """Verdict each claim as true, false, or half-true. Respond with only the verdict.

Here are some examples:
Claim: "Adobe's Firefly AI image generator was exclusively trained on content from Adobe Stock and public domain images." → true
Claim: "Midjourney's AI models are trained only on images explicitly licensed from artists, without any web scraping." → false
Claim: "Canva's AI image generation features solely use publicly licensed data for training to avoid copyright issues." → half-true

Claim: "{claim}" →""".format(claim=claim)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text.strip().lower()

few_shot = [verdict_few_shot(c["text"]) for c in claims]
print("Few-shot results:", few_shot)

Few-shot results: ['true', 'false', 'half-true', 'false', 'false', 'true', 'false', 'true', 'false', 'half-true', 'false']


In [ ]:
# Approach 3 - Context-engineered prompt
import json

FACTCHECK_PROMPT = """## System Instruction
You are a fact-checker for a news organization. Your job is to
evaluate claims for accuracy. You are cautious: when you aren't
sure, you say so. You never fabricate citations or studies.

## Task
Evaluate the claim below. Consider common misconceptions,
frequently misquoted statistics, and whether the claim could
be technically true but misleading.

## Output Format
Respond with ONLY valid JSON:
{{"verdict": "true" | "false" | "half-true",
  "confidence": "high" | "medium" | "low",
  "reasoning": "2-3 sentences explaining your verdict",
  "red_flags": "any warning signs you noticed, or 'none'"}}

## Calibration Examples
Claim: "Adobe's Firefly AI image generator was exclusively trained on content from Adobe Stock and public domain images."
{{"verdict": "true", "confidence": "high",
  "reasoning": "Adobe has publicly stated and emphasized that Firefly is trained on Adobe Stock content, openly licensed works, and public domain content to avoid copyright infringement concerns.",
  "red_flags": "none"}}

Claim: "Midjourney's AI models are trained only on images explicitly licensed from artists, without any web scraping."
{{"verdict": "false", "confidence": "high",
  "reasoning": "Midjourney, like many other generative AI models, has faced criticism and lawsuits regarding its training data, which is widely believed to include scraped internet images without explicit artist consent or licensing.",
  "red_flags": "The claim uses absolutes ('only on images explicitly licensed') which are often red flags in complex data sourcing issues."}}

Claim: "Canva's AI image generation features solely use publicly licensed data for training to avoid copyright issues."
{{"verdict": "half-true", "confidence": "medium",
  "reasoning": "While Canva aims to use publicly licensed data, it's difficult to guarantee 'solely' without deep insight into their exact data sources, and the nuances of licensing can be complex.",
  "red_flags": "The use of 'solely' often indicates a claim that might be technically true but have caveats or exceptions."}}

Claim: "GPT-4 was trained using RLHF."
{{"verdict": "true", "confidence": "high",
  "reasoning": "OpenAI has confirmed that GPT-4, like its predecessors, utilizes Reinforcement Learning from Human Feedback (RLHF) as a key part of its training and alignment process to improve its responses.",
  "red_flags": "none"}}

## Constraints
- If you genuinely don't know whether a claim is accurate, use
  verdict "half-true" with confidence "low" and explain what
  you'd need to verify it.
- Never invent a source to support your verdict.

## Claim to Evaluate
{claim}"""

def verdict_context(claim):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=FACTCHECK_PROMPT.format(claim=claim)
    )
    try:
        # Strip markdown code block delimiters if present
        clean_response_text = response.text.strip()
        if clean_response_text.startswith("```json"):
            clean_response_text = clean_response_text[7:]  # Remove ```json
        if clean_response_text.endswith("```"):
            clean_response_text = clean_response_text[:-3]  # Remove ```

        result = json.loads(clean_response_text)
        return result # Return the full JSON object
    except json.JSONDecodeError:
        return {"verdict": "parse_error", "reasoning": "JSON parsing failed", "confidence": "low", "red_flags": "invalid_json_format"}

context_results = [verdict_context(c["text"]) for c in claims]
print("Context-engineered prompt results:", context_results)

Context-engineered prompt results: [{'verdict': 'true', 'confidence': 'high', 'reasoning': 'OpenAI has confirmed that GPT-4, like its predecessors, utilizes Reinforcement Learning from Human Feedback (RLHF) as a key part of its training and alignment process to improve its responses.', 'red_flags': 'none'}, {'verdict': 'true', 'confidence': 'high', 'reasoning': 'Adobe has consistently and publicly stated that Firefly is trained on content from Adobe Stock, openly licensed works, and public domain content. This strategy aims to differentiate Firefly and mitigate copyright concerns.', 'red_flags': 'none'}, {'verdict': 'half-true', 'confidence': 'medium', 'reasoning': "While Canva has publicly stated its commitment to using ethically sourced training data, including content from its own library, licensed content, and public domain materials, guaranteeing that it 'solely' uses such data for all its AI image generation features (especially when partnering with other model providers) is diff

In [ ]:
# Approach 4 - LLM-assisted refinement
errors = []
for c, pred in zip(claims, context_results):
    if pred != c["truth"]:
        errors.append(
            f'Claim: "{c["text"]}"\n'
            f'  Your verdict: {pred}\n'
            f'  Correct verdict: {c["truth"]}'
        )

error_report = "\n\n".join(errors) if errors else "All verdicts matched."

meta_prompt = f"""You are a prompt optimization specialist. Below is a\nfact-checking prompt and the claims it got wrong.\n\n## Current Prompt\n{FACTCHECK_PROMPT}\n\n## Incorrect Verdicts\n{error_report}\n\n## Your Task\n1. Diagnose why each misclassification happened.\n2. Propose a revised prompt that fixes these errors without\n   breaking the claims that already work.\n3. Return ONLY the complete revised prompt."""

refinement = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=meta_prompt
)
print("=== SUGGESTED REFINEMENTS ===")
print(refinement.text)

# Copy the refined prompt into a new function and run it:
# refined_results = [verdict_refined(c["text"]) for c in claims]

=== SUGGESTED REFINEMENTS ===
Here's the diagnosis of the misclassifications and the revised prompt.

---

## Diagnosis of Misclassifications

The prompt's original output was mostly accurate across the board, with only two claims being misclassified.

1.  **Claim: "Adobe guarantees that Firefly users will not face copyright infringement claims for images generated with its tool."**
    *   **Original Verdict (Prompt):** `half-true`
    *   **Correct Verdict:** `true`
    *   **Diagnosis:** The prompt over-indexed on the absolute term "will not face." While it's technically true that anyone can *file* a claim, Adobe's indemnification is a strong, legally binding guarantee against *successful* claims and the financial/legal burden of defense for valid usage. The prompt interpreted "guarantee" too literally (as preventing the *act* of a claim being filed) rather than in the practical, industry-standard sense of robust legal and financial protection. The existing `red_flags` guidance rega

In [ ]:
# LLM Refined Approach - Context-engineered prompt
import json

FACTCHECK_PROMPT = """## System Instruction
You are a fact-checker for a news organization. Your job is to
evaluate claims for accuracy. You are cautious: when you aren't
sure, you say so. You never fabricate citations or studies.

## Task
Evaluate the claim below. Consider common misconceptions,
frequently misquoted statistics, and whether the claim could
be technically true but misleading.

## Output Format
Respond with ONLY valid JSON:
{{"verdict": "true" | "false" | "half-true",
  "confidence": "high" | "medium" | "low",
  "reasoning": "2-3 sentences explaining your verdict",
  "red_flags": "any warning signs you noticed, or 'none'"}}

## Calibration Examples
Claim: "GPT-4 was trained using RLHF."
{{"verdict": "true", "confidence": "high",
  "reasoning": "OpenAI has confirmed that GPT-4, like its predecessors, utilizes Reinforcement Learning from Human Feedback (RLHF) as a key part of its training and alignment process to improve its responses.",
  "red_flags": "none"}}

Claim: "Major airlines guarantee a refund if they cancel a flight."
{{"verdict": "true", "confidence": "high",
  "reasoning": "Airlines are legally mandated and widely known to offer refunds for flights they cancel, though the specific process and timing may vary. The term 'guarantee' here refers to a firm commitment and legal obligation, not an absolute prevention of any minor issue or delay in receiving the refund.",
  "red_flags": "none"}}

Claim: "Adobe's Firefly AI image generator was exclusively trained on content from Adobe Stock and public domain images."
{{"verdict": "true", "confidence": "high",
  "reasoning": "Adobe has publicly stated and emphasized that Firefly is trained on Adobe Stock content, openly licensed works, and public domain content to avoid copyright infringement concerns.",
  "red_flags": "none"}}

Claim: "Midjourney's AI models are trained only on images explicitly licensed from artists, without any web scraping."
{{"verdict": "false", "confidence": "high",
  "reasoning": "Midjourney, like many other generative AI models, has faced criticism and lawsuits regarding its training data, which is widely believed to include scraped internet images without explicit artist consent or licensing.",
  "red_flags": "The claim uses absolutes ('only on images explicitly licensed') which are often red flags in complex data sourcing issues."}}

Claim: "Canva's AI image generation features solely use publicly licensed data for training to avoid copyright issues."
{{"verdict": "half-true", "confidence": "medium",
  "reasoning": "While Canva aims to use publicly licensed data, it's difficult to guarantee 'solely' without deep insight into their exact data sources, and the nuances of licensing can be complex.",
  "red_flags": "The use of 'solely' often indicates a claim that might be technically true but have caveats or exceptions."}}

Claim: "This AI model's training data exclusively consists of scientific papers and academic journals."
{{"verdict": "half-true", "confidence": "high",
  "reasoning": "While the AI model may indeed incorporate a significant amount of scientific papers and academic journals into its training data (making part of the claim true), it is highly improbable for a large-scale AI model's training data to *exclusively* consist of only these types of documents, as most models draw from a much broader and diverse range of internet content to achieve general capabilities.",
  "red_flags": "The absolute term 'exclusively consists of' is a major red flag, as large AI models almost always use highly diverse datasets."}}

## Constraints
- If you genuinely don't know whether a claim is accurate, use
  verdict "half-true" with confidence "low" and explain what
  you'd need to verify it.
- Never invent a source to support your verdict.

## Claim to Evaluate
{claim}"""

def verdict_context(claim):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=FACTCHECK_PROMPT.format(claim=claim)
    )
    try:
        # Strip markdown code block delimiters if present
        clean_response_text = response.text.strip()
        if clean_response_text.startswith("```json"):
            clean_response_text = clean_response_text[7:]  # Remove ```json
        if clean_response_text.endswith("```"):
            clean_response_text = clean_response_text[:-3]  # Remove ```

        result = json.loads(clean_response_text)
        return result # Return the full JSON object
    except json.JSONDecodeError:
        return {"verdict": "parse_error", "reasoning": "JSON parsing failed", "confidence": "low", "red_flags": "invalid_json_format"}

context_results = [verdict_context(c["text"]) for c in claims]
print("Context-engineered prompt results:", context_results)

Context-engineered prompt results: [{'verdict': 'true', 'confidence': 'high', 'reasoning': 'OpenAI has confirmed that GPT-4, like its predecessors, utilizes Reinforcement Learning from Human Feedback (RLHF) as a key part of its training and alignment process to improve its responses.', 'red_flags': 'none'}, {'verdict': 'false', 'confidence': 'high', 'reasoning': 'Adobe has publicly stated that its Firefly AI image generator is trained on a dataset comprising Adobe Stock content, public domain content, and *openly licensed content*. The claim is false because it incorrectly asserts that Firefly was *exclusively* trained on only Adobe Stock and public domain images, omitting the third significant category of openly licensed works.', 'red_flags': "The use of the absolute term 'exclusively' often signals that a claim might be too narrow or omit important details."}, {'verdict': 'half-true', 'confidence': 'medium', 'reasoning': "While Canva has publicly stated its commitment to using ethica

In [ ]:
# Comparison table
# Removed Context Results due to parsing and format issues to be resolved in next cell
print(f"{'Claim':<55} {'Truth':<12} {'Zero':<12} {'Few':<12}")
print("-" * 91)
for c, zs, fs in zip(claims, zero_shot, few_shot):
    short = c["text"][:52] + "..." if len(c["text"]) > 55 else c["text"]
    print(f"{short:<55} {c['truth']:<12} {zs:<12} {fs:<12}")

# Accuracy per approach
print("\n--- Accuracy (Zero-shot & Few-shot) ---")
for name, results in [("Zero-shot", zero_shot), ("Few-shot", few_shot)]:
    correct = sum(1 for c, r in zip(claims, results) if c["truth"] == r)
    print(f"{name}: {correct}/{len(claims)} ({100*correct/len(claims):.0f}%)")

Claim                                                   Truth        Zero         Few         
-------------------------------------------------------------------------------------------
GPT-4 was trained using RLHF.                           true         true         true        
Adobe's Firefly AI image generator was exclusively t... true         false        false       
Canva's AI image generation features solely use publ... half-true    false        half-true   
Midjourney's AI models are trained only on images ex... false        false        false       
Google's Imagen AI models are trained entirely on in... false        false        false       
Companies like Stability AI (Stable Diffusion) have ... true         true         true        
Most major AI image generators explicitly state that... false        false        false       
Private libraries of licensed assets are generally p... true         true         true        
The ethics of using internet-scraped data for AI tra.

In [ ]:
# Comprehensive Comparison table
print(f"{'Claim':<55} {'Truth':<12} {'Zero':<12} {'Few':<12} {'Context':<12} {'Refined':<12}")
print("-" * 115) # Adjusted length for new column
for c, zs, fs, ce_dict, re_dict in zip(claims, zero_shot, few_shot, context_results, refined_results):
    short = c["text"][:52] + "..." if len(c["text"]) > 55 else c["text"]
    context_verdict = ce_dict['verdict'] if isinstance(ce_dict, dict) and 'verdict' in ce_dict else 'N/A'
    refined_verdict = re_dict['verdict'] if isinstance(re_dict, dict) and 'verdict' in re_dict else 'N/A'
    print(f"{short:<55} {c['truth']:<12} {zs:<12} {fs:<12} {context_verdict:<12} {refined_verdict:<12}")

print("\n--- Accuracy per approach ---")
for name, results_list in [("Zero-shot", zero_shot), ("Few-shot", few_shot),
                            ("Context", context_results), ("Refined", refined_results)]:
    if name in ["Zero-shot", "Few-shot"]:
        correct = sum(1 for c, r in zip(claims, results_list) if c["truth"] == r)
    else: # For Context and Refined results, which are dictionaries
        correct = sum(1 for c, r_dict in zip(claims, results_list) if isinstance(r_dict, dict) and 'verdict' in r_dict and c["truth"] == r_dict["verdict"])
    print(f"{name}: {correct}/{len(claims)} ({100*correct/len(claims):.0f}%)")

Claim                                                   Truth        Zero         Few          Context      Refined     
-------------------------------------------------------------------------------------------------------------------
GPT-4 was trained using RLHF.                           true         true         true         true         true        
Adobe's Firefly AI image generator was exclusively t... true         false        false        false        half-true   
Canva's AI image generation features solely use publ... half-true    false        half-true    half-true    half-true   
Midjourney's AI models are trained only on images ex... false        false        false        false        false       
Google's Imagen AI models are trained entirely on in... false        false        false        false        false       
Companies like Stability AI (Stable Diffusion) have ... true         true         true         true         true        
Most major AI image generators explic

###**Reflection**

Reviewing the red flags in the context engineered responses helped highlight how certain wording choices, especially absolute terms like “exclusively,” “only,” or “strictly,” can affect how a claim’s accuracy is evaluated. The model consistently identified these types of terms and sometimes assigned different ratings than expected based on how strongly the wording limited the claim. For example, some claims that were rated as true were instead marked as half true because the language suggested a level of certainty that was not fully supported. In other cases, claims that seemed partially accurate were labeled false when the absolute wording could not be supported. This process showed how AI can be useful for identifying patterns in language that might otherwise be overlooked. At the same time, the model’s strict interpretation of absolute wording did not always reflect the more practical or contextual meaning of the claims, which shows that human judgment still plays an important role in evaluating accuracy when using AI tools.